# Appliance Energy Forecasting — Part 4: SARIMAX

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/appliance-energy-forecasting"
os.chdir(PROJECT_ROOT)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams["figure.figsize"] = (14, 5)
np.random.seed(0)

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/forecasts", exist_ok=True)
os.makedirs("outputs/metrics", exist_ok=True)
os.makedirs("outputs/model_objects", exist_ok=True)


In [ ]:
hourly = pd.read_csv("data/processed/appliance_hourly.csv", index_col=0, parse_dates=True)
y = hourly["Appliances"]

TARGET = "Appliances"
HORIZON = 24
DAILY_PERIOD = 24
TEST_STEPS = 14 * 24

train = y.iloc[:-TEST_STEPS]
test = y.iloc[-TEST_STEPS:]

print("train:", train.shape, "test:", test.shape)


In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mase(y_true, y_pred, y_train, seasonality=24):
    y_train = pd.Series(y_train).astype(float)
    seasonal_errors = np.abs(y_train.iloc[seasonality:].values - y_train.iloc[:-seasonality].values)
    scale = seasonal_errors.mean()
    if scale == 0:
        return np.nan
    return np.mean(np.abs(y_true - y_pred)) / scale

def evaluate_forecast(name, y_true, y_pred, y_train):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred, index=y_true.index).astype(float)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MASE": mase(y_true, y_pred, y_train, seasonality=DAILY_PERIOD),
        "Bias": np.mean(y_pred - y_true),
    }


### AIC grid search over p, d, q

Seasonal order is fixed at `(1, 1, 1, 24)` based on the ACF/PACF and stationarity results
from notebook 1 (seasonal differencing at lag 24 was required). Searching seasonal P, D, Q
as well would multiply runtime by orders of magnitude for hourly data, so only the
non-seasonal p, d, q grid specified in the assignment (0-6, 0-2, 0-6) is searched here.

This search fits on a **subsample** of the training data (last 60 days) to keep runtime
reasonable in Colab. The final chosen model is then refit on the full training set.

Expect this cell to take several minutes to run.

In [ ]:
SEARCH_SAMPLE_DAYS = 60
search_train = train.iloc[-SEARCH_SAMPLE_DAYS*24:]

p_values = range(0, 7)
d_values = range(0, 3)
q_values = range(0, 7)

results = []
for p, d, q in itertools.product(p_values, d_values, q_values):
    try:
        model = SARIMAX(
            search_train,
            order=(p, d, q),
            seasonal_order=(1, 1, 1, 24),
            trend="c" if d == 0 else None,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = model.fit(disp=False, maxiter=50)
        results.append({"p": p, "d": d, "q": q, "aic": fit.aic})
    except Exception:
        continue

aic_results = pd.DataFrame(results).sort_values("aic").reset_index(drop=True)
aic_results.to_csv("outputs/metrics/sarimax_aic_search.csv", index=False)
print(aic_results.head(10))


In [ ]:
best_order = tuple(aic_results.iloc[0][["p", "d", "q"]].astype(int))
print("best order (p, d, q):", best_order)


### Refit best model on full training set with exogenous variables

Candidate exogenous variables are outdoor weather columns available in the dataset.
These are included because outdoor conditions plausibly relate to heating/cooling appliance
use, but note: at forecast time these must either be known in advance or forecast themselves,
which is addressed in the report questions.

In [ ]:
candidate_exog_cols = ["T_out", "RH_out", "Windspeed", "Visibility", "Tdewpoint"]
exog_cols = [c for c in candidate_exog_cols if c in hourly.columns]
print("exogenous columns used:", exog_cols)

X = hourly[exog_cols]
X_train = X.iloc[:-TEST_STEPS]
X_test = X.iloc[-TEST_STEPS:]

sarimax_model = SARIMAX(
    train,
    exog=X_train,
    order=best_order,
    seasonal_order=(1, 1, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_fit = sarimax_model.fit(disp=False, maxiter=100)
print(sarimax_fit.summary())


### Residual diagnostics

Checks whether the model has captured the autocorrelation structure in the data.
A well-fitted model should leave residuals that resemble white noise:
- ACF of residuals should show no significant spikes
- Residual distribution should be roughly symmetric/normal
- Ljung-Box test: p > 0.05 means no significant autocorrelation remains in residuals

In [ ]:
residuals = sarimax_fit.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_acf(residuals.dropna(), lags=48, ax=axes[0])
axes[0].set_title("ACF of SARIMAX Residuals")
axes[1].hist(residuals.dropna(), bins=50)
axes[1].set_title("Distribution of SARIMAX Residuals")
plt.tight_layout()
plt.savefig("outputs/figures/07_sarimax_residual_diagnostics.png", dpi=150)
plt.show()

lb_test = acorr_ljungbox(residuals.dropna(), lags=[24], return_df=True)
print(lb_test)


### Rolling 24h forecast across the test period, with 95% confidence intervals

In [ ]:
def rolling_sarimax_forecast(fit_result, order, seasonal_order, y_train, y_test, X_train, X_test, horizon=24):
    history_y = y_train.copy()
    history_X = X_train.copy() if X_train is not None else None
    preds, lower, upper = [], [], []

    for start in range(0, len(y_test), horizon):
        block_index = y_test.index[start:start + horizon]
        block_horizon = len(block_index)

        model = SARIMAX(
            history_y,
            exog=history_X,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        fit = model.fit(disp=False, maxiter=50)

        exog_block = X_test.iloc[start:start + block_horizon] if X_test is not None else None
        fc = fit.get_forecast(steps=block_horizon, exog=exog_block)

        preds.append(pd.Series(fc.predicted_mean.values, index=block_index))
        ci = fc.conf_int(alpha=0.05)
        lower.append(pd.Series(ci.iloc[:, 0].values, index=block_index))
        upper.append(pd.Series(ci.iloc[:, 1].values, index=block_index))

        history_y = pd.concat([history_y, y_test.iloc[start:start + block_horizon]])
        if history_X is not None:
            history_X = pd.concat([history_X, X_test.iloc[start:start + block_horizon]])

    return pd.concat(preds), pd.concat(lower), pd.concat(upper)

sarimax_pred, sarimax_lower, sarimax_upper = rolling_sarimax_forecast(
    sarimax_fit, best_order, (1, 1, 1, 24), train, test, X_train, X_test, horizon=HORIZON
)


**Note on runtime:** this refits SARIMAX for every 24h block across the 14-day test
period (14 refits total), which is more realistic than forecasting the whole 14 days from
a single fit, but takes longer. If this is too slow in your Colab session, reduce `horizon`
refits by forecasting the full `TEST_STEPS` in one call instead — mention this tradeoff in
the report if you do.

In [ ]:
sarimax_results = evaluate_forecast("sarimax", test, sarimax_pred.reindex(test.index), train)
print(sarimax_results)


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
plot_window = test.index[:24*7]

test.loc[plot_window].plot(ax=ax, label="actual", color="black", linewidth=2)
sarimax_pred.reindex(plot_window).plot(ax=ax, label="sarimax forecast", color="tab:blue")
ax.fill_between(
    plot_window,
    sarimax_lower.reindex(plot_window),
    sarimax_upper.reindex(plot_window),
    color="tab:blue", alpha=0.2, label="95% CI"
)
ax.set_title("SARIMAX Forecast vs Actual - First 7 Days of Test Period")
ax.set_ylabel("Appliances (Wh)")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/figures/08_sarimax_forecast.png", dpi=150)
plt.show()


In [ ]:
sarimax_forecast_df = pd.DataFrame({
    "actual": test,
    "sarimax": sarimax_pred.reindex(test.index),
    "sarimax_lower": sarimax_lower.reindex(test.index),
    "sarimax_upper": sarimax_upper.reindex(test.index),
})
sarimax_forecast_df.to_csv("outputs/forecasts/sarimax_forecast.csv")

pd.DataFrame([sarimax_results]).to_csv("outputs/metrics/sarimax_metrics.csv", index=False)
print("saved sarimax forecast and metrics")
